In [ ]:
import sys
print(sys.executable)

# Airline Passenger Satisfaction Analysis

Machine learning analysis using PCA and Logistic Regression to understand key drivers of airline passenger satisfaction.

## Problem Statement

Airlines collect passenger feedback on multiple service aspects such as seating comfort, onboard service, and flight delays. Understanding which factors influence passenger satisfaction can help airlines improve customer experience.

This project applies machine learning techniques to analyze airline passenger satisfaction and identify the most influential factors affecting satisfaction levels.

In [ ]:
import pandas as pd
import numpy as np 

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, accuracy_score, confusion_matrix, roc_curve, auc

from sklearn.linear_model import LinearRegression, LogisticRegression

from sklearn.decomposition import PCA 

from sklearn.preprocessing import StandardScaler, LabelEncoder

from sklearn.model_selection import train_test_split

from matplotlib import pyplot as plt

## Dataset Overview

In [ ]:
df = pd.read_csv("/Users/mehedihasan/Downloads/airline passenger satisfaction/airline_passenger_satisfaction.csv")
print(df.head())
df.info()
df.describe()

## Missing Value Investigation

In [ ]:
print(df.isnull().sum())

missing_values = df[df['Arrival Delay'].isnull()]
print(missing_values)

subset = df[df["Arrival Delay"].isnull()][["Departure Delay", "Flight Distance"]]
print(subset)

## Predicting Missing Arrival Delay using Linear Regression

In [ ]:
df_known = df[df['Arrival Delay'].notnull()]
df_missing = df[df['Arrival Delay'].isnull()]

X_train = df_known[['Departure Delay', 'Flight Distance']]
Y_train = df_known["Arrival Delay"]

x_predict = df_missing[["Departure Delay", 'Flight Distance']]

model = LinearRegression()
model.fit(X_train, Y_train)

predicted_values = model.predict(x_predict)

df.loc[df_missing.index, "Arrival Delay"] = predicted_values

## Linear Regression Evaluation

In [ ]:
points = plt.scatter(Y_train, model.predict(X_train), alpha=0.5)

line_x = Y_train.min(), Y_train.max()
line_y = Y_train.min(), Y_train.max()

plt.plot(line_x, line_y)

plt.xlabel("Actual Arrival Delay")
plt.ylabel("Predicted Arrival Delay")
plt.title("Prediction vs Actual")

plt.show()

## The error matrices

In [ ]:
y_predicted = model.predict(X_train)
y_true = Y_train

mae = mean_absolute_error(y_predicted, y_true)
mse = mean_squared_error(y_predicted, y_true)
r2 = r2_score(y_true, y_predicted)

print(mae, mse, r2)

## PCA Analysis of Service Features

In [ ]:
service_columns = [
"Departure and Arrival Time Convenience",
"Ease of Online Booking",
"Check-in Service",
"Online Boarding",
"Gate Location",
"On-board Service",
"Seat Comfort",
"Leg Room Service",
"Cleanliness",
"Food and Drink",
"In-flight Service",
"In-flight Wifi Service",
"In-flight Entertainment",
"Baggage Handling"
]

df_service = df[service_columns]

## Scaling Value for PCA

In [ ]:
scaler = StandardScaler()

scaled = scaler.fit_transform(df_service)

## PCA

In [ ]:
pca = PCA()

pc_values = pca.fit_transform(scaled)

print(pca.explained_variance_ratio_)

## PCA Visualization

In [ ]:
PC_1 = pc_values[:,0]
PC_2 = pc_values[:,1]

temp_dict = {
"Satisfied":"green",
"Neutral or Dissatisfied":"red"
}

df["satisfaction_color"] = df["Satisfaction"].map(temp_dict)

plt.scatter(PC_1, PC_2, color=df["satisfaction_color"], alpha=0.4)

plt.xlabel("Service Quality")
plt.ylabel("Pre-boarding Experience")

plt.title("Passenger Satisfaction PCA Visualization")

plt.show()

## Logistic Regression using Flight Delays

In [ ]:
le = LabelEncoder()
df['target'] = le.fit_transform(df["Satisfaction"])

x = df[["Arrival Delay", "Departure Delay"]]
y = df['target']

##Split
x_train, x_test, y_train, y_test = train_test_split(
x, y, test_size=0.2, random_state=42, stratify=y
)

##Scaling

scale = StandardScaler()

train_scale = scale.fit_transform(x_train)
test_scale = scale.transform(x_test)

## Model

In [ ]:
log_reg_delay = LogisticRegression()

log_reg_delay.fit(train_scale,y_train)

prediction = log_reg_delay.predict(test_scale)

## Model Evaluation with Confusion matrix

In [ ]:
acc_score = accuracy_score(y_test, prediction)
cm = confusion_matrix(y_test, prediction)

print("Accuracy:", acc_score)
print(cm)

## Final Logistic Regression Model

In [ ]:
df["PC_1"] = PC_1
df["PC_2"] = PC_2

df["Customers"] = le.fit_transform(df["Customer Type"])

df = pd.get_dummies(df, columns=["Class"], drop_first=True)

###Features

X_reg = df[
["Arrival Delay","Departure Delay","PC_1","PC_2","Customers","Class_Economy","Class_Economy Plus"]
]

Y_reg = df["target"]

##Train_test division

X_reg_train, X_reg_test, Y_reg_train, Y_reg_test = train_test_split(
X_reg, Y_reg,
test_size=0.2,
random_state=42,
stratify=Y_reg
)

##Model itself

log_reg_full = LogisticRegression()

log_reg_full.fit(X_reg_train, Y_reg_train)

full_prediction = log_reg_full.predict(X_reg_test)

columns = log_reg_full.feature_names_in_

full_coeff = log_reg_full.coef_.flatten()


## Evaluation Final logistic regression model


In [ ]:
final_acc_score = accuracy_score(Y_reg_test,full_prediction)
final_cm = confusion_matrix(Y_reg_test,full_prediction)
print("Model performance:",final_acc_score)
print("Confusion matrics:", final_cm)

## Coefficients Bar Chart (Plot evaluation)


In [ ]:
Final_X_values = ["Arrival Delay", 'Departure Delay', "PC_1", "PC_2", "Customers", 'Class_Economy', 'Class_Economy Plus']
Final_y_values = full_coeff
bars = plt.bar(Final_X_values, Final_y_values, 0.8)
base_line_final =plt.axhline(y=0, color="Black", linestyle='--', linewidth=1.5)
plt.xlabel("Features Type")
plt.ylabel("Feature's Coefficients")
plt.title("Final Logistic regression Coefficients Plot")
plt.xticks(rotation=45)
plt.show()

## Feature importance Ranking chart(Sorted coefficients)


In [ ]:
pairs = list(zip(columns, full_coeff))
sorting_pairs = sorted(pairs, key=lambda x: x[1], reverse=True)

for f, coeff in sorting_pairs:
    print((f'{f} {round(coeff,3)}'))

## AUC-ROC curve for Classification model


In [ ]:
y_pred_prob = log_reg_full.predict_proba(X_reg_test)[:, 1]
fpr, tpr, thresholds = roc_curve(Y_reg_test, y_pred_prob)
roc_auc= auc(fpr, tpr)

## Plotting Roc curve

plt.figure()
plt.plot(fpr, tpr, lw =2, label=f'ROC curve (area)=, {roc_auc:.2f}')
plt.plot([0,1],[0,1], color="Navy", linestyle='--', label="random guess")
plt.xlim([0,1])
plt.ylim([0,1])
plt.xlabel("False positive rate")
plt.ylabel("True positive rate")
plt.title("Receiver Operating Characteristic (ROC) Curve")
plt.legend(loc="lower right")
plt.show()

## Key Findings

Flight class has the strongest influence on satisfaction
• Customer type strongly affects satisfaction probability
• Service quality factors captured by PCA significantly impact satisfaction
• Flight delays have weaker impact compared to service quality